# Europe's temperature outlook

Compare next week's temperatures across Europe, then click a place to see its
forecast range. The page checks for a recent ECMWF run each time you open it.

In [ ]:
from pathlib import Path
import sys
from manywidgets import Fullscreen

examples = next(
    p for p in [Path("."), Path("docs/examples")]
    if (p / "forecast/widget.ts").exists()
)
sys.path.insert(0, str(examples.resolve()))
from forecast import EuropeForecast

forecast = EuropeForecast()
Fullscreen(forecast, widget_id="europe-forecast")

## Put it in a notebook

This example wraps a browser-side forecast map in `Fullscreen`. The calendar,
map tabs and local chart share the same widget state. Dates and weather data are
loaded when the widget opens, so the exported page keeps working without a kernel.

Run `npm run build:forecast` from the repository root before opening this notebook.
The renderer uses deck.gl and MapLibre, with ZarrLayer from
[deck.gl-raster](https://developmentseed.org/deck.gl-raster/) reading the
[Dynamical archive](https://dynamical.org/catalog/ecmwf-ifs-ens-forecast-15-day-0-25-degree/)
directly. The basemap comes from [OpenFreeMap](https://openfreemap.org/quick_start/).

The widget exposes `day_index`, `metric` (`mean` or `spread`) and `playing`.
These are ordinary traits that can be linked to other manywidgets controls.

```python
from pathlib import Path
import sys
from manywidgets import Fullscreen

examples = next(
    p for p in [Path("."), Path("docs/examples")]
    if (p / "forecast/widget.ts").exists()
)
sys.path.insert(0, str(examples.resolve()))
from forecast import EuropeForecast

forecast = EuropeForecast()
Fullscreen(forecast, widget_id="europe-forecast")
```

The wrapper lives in `forecast/__init__.py` beside the renderer. To use a smaller
region, pass bounds and locations when creating the widget:

```python
forecast = EuropeForecast(
    region_name="Switzerland",
    bounds=[5.9, 45.8, 10.6, 47.9],
    locations=[
        {"name": "Zurich", "lon": 8.542, "lat": 47.377},
        {"name": "Geneva", "lon": 6.143, "lat": 46.204},
    ],
    initial_location="Zurich",
    forecast_days=7,
    day_index=0,
    temperature_range=[-5, 25],
    spread_range=[0, 6],
)
Fullscreen(forecast)
```

Bounds, locations, forecast length and palettes are set at creation. Create a new
widget to change them. `day_index`, `metric` and `playing` can change while it is
open. The data reader still uses the same ECMWF archive; this is not an arbitrary
Zarr viewer.

To read a compatible copy of the ECMWF archive, pass
`store_url="https://your-host.example/ecmwf.zarr"` at creation. The URL must point
to the Zarr root and allow browser access. The copy must retain the original
variables, units, coordinates, dimensions and chunks, and contain recent runs.
Other Zarr layouts need changes to the data reader.

The first map load reads several ensemble chunks, so it can take a while on a
slow connection. Once loaded, changing the date or map layer reuses the data.
The browser needs WebGL2 and access to the public data services.

[Example documentation](https://github.com/developmentseed/manywidgets/tree/main/docs/examples/forecast)
covers setup, code structure and data limitations.